# EDA (AlphaEarth)

This notebook contains exploratory analysis / target distribution checks for the AlphaEarth embeddings experiment.

Split out of the prior `01_model_evaluation.ipynb`.

In [ ]:
# If you want to (re)generate the report figures, run this notebook from
# the `alphaearth_src/notebooks` directory so relative paths resolve.

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Publication-ish theme: light major y-grid only, no minor grid
sns.set_theme(
    style="white",
    font_scale=1.05,
    rc={
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "0.88",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
    },
)


## 1. Load embeddings and modeling sample

Same filters as training: nonzero `log_popden` and `log_POBTOT`, embeddings + state dummies.


In [ ]:
print("Loading data...")
file_path = '../data/embeddings/00_all_mexico_embeddings_combined.csv'
df = pd.read_csv(file_path)

state_dummies = pd.get_dummies(df['CVE_ENT'], prefix='ENT', drop_first=True)
embedding_cols = [f'A{i:02d}' for i in range(64)]

X_base = df[embedding_cols]
# Columns log_POBTOT / log_popden store natural log (ln); back-transform with np.exp(...)
y_log_pobtot = df["log_POBTOT"]
y_log_popden = df["log_popden"]

X_base = X_base.replace([np.inf, -np.inf], np.nan).fillna(0)
y_log_pobtot = y_log_pobtot.replace([np.inf, -np.inf], np.nan).fillna(0)
y_log_popden = y_log_popden.replace([np.inf, -np.inf], np.nan).fillna(0)

# Filter invalid / placeholder rows (ln targets masked as 0)
mask = (y_log_popden != 0) & (y_log_pobtot != 0)

X_filt = X_base[mask].copy()
state_dummies_filt = state_dummies[mask].copy()

# Combine embeddings and state dummies
X = pd.concat([X_filt, state_dummies_filt], axis=1)
y_den = y_log_popden[mask].copy()

print(f"Dataset size after filtering: {len(X):,} rows")
print(f"Number of features: {X.shape[1]}")


## 1.5 ln-scale targets and filtering

Summary table (raw vs filtered), ln(popden) histograms/KDE, counts, ECDF on original scale, ln(population), and an optional urban/rural grouping column if present.


In [ ]:
# Publication-style defaults
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

FIG_DIR = os.path.abspath(os.path.join("..", "..", "..", "report", "figs"))
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name: str):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, facecolor="white")
    return path

# Basic columns check
required = ["log_POBTOT", "log_popden"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing expected columns: {missing}. Available columns include: {list(df.columns)[:20]} ...")

# Reconstruct the filtering mask exactly as used later
mask = (df["log_popden"].replace([np.inf, -np.inf], np.nan).fillna(0) != 0) & (
    df["log_POBTOT"].replace([np.inf, -np.inf], np.nan).fillna(0) != 0
)

summary = pd.DataFrame({
    "rows": [len(df), int(mask.sum())],
    "share_of_total": [1.0, float(mask.mean())],
    "min_log_popden": [df["log_popden"].replace([np.inf, -np.inf], np.nan).min(), df.loc[mask, "log_popden"].min()],
    "p50_log_popden": [df["log_popden"].replace([np.inf, -np.inf], np.nan).median(), df.loc[mask, "log_popden"].median()],
    "p99_log_popden": [df["log_popden"].replace([np.inf, -np.inf], np.nan).quantile(0.99), df.loc[mask, "log_popden"].quantile(0.99)],
}, index=["Raw", "Filtered"])

display(summary)

# 1) ln(popden) distribution (raw vs filtered)
# NOTE: "Density" here means *probability density* (area under histogram integrates to 1).
vals_raw = df["log_popden"].replace([np.inf, -np.inf], np.nan).dropna()
vals_f = df.loc[mask, "log_popden"].replace([np.inf, -np.inf], np.nan).dropna()

raw_color = "#5f6368"       # darker gray
filt_color = "#1f77b4"      # blue

fig, ax = plt.subplots(figsize=(7.2, 3.9))
sns.histplot(
    vals_raw,
    bins=80,
    stat="density",
    color=raw_color,
    alpha=0.30,
    label="Raw",
    ax=ax,
)
sns.kdeplot(vals_f, color=filt_color, linewidth=2.2, label="Filtered", ax=ax)
ax.set_title("Population density target distribution (ln-scale)", fontweight="bold")
ax.set_xlabel("ln(population density)")
ax.set_ylabel("Probability density")
ax.grid(True, axis="y")
ax.grid(False, axis="x")
ax.legend(frameon=True)
plt.tight_layout()
path1 = savefig("eda_log_popden_distribution.png")
plt.show()
print("Saved:", path1)

# 1b) Same histogram, but showing *raw counts* for interpretability
fig, ax = plt.subplots(figsize=(7.2, 3.9))
sns.histplot(
    vals_raw,
    bins=80,
    stat="count",
    color=raw_color,
    alpha=0.30,
    label="Raw",
    ax=ax,
)
sns.histplot(
    vals_f,
    bins=80,
    stat="count",
    color=filt_color,
    alpha=0.35,
    label="Filtered",
    ax=ax,
)
ax.set_title("Population density target distribution (ln-scale) — counts", fontweight="bold")
ax.set_xlabel("ln(population density)")
ax.set_ylabel("Count")
ax.grid(True, axis="y")
ax.grid(False, axis="x")
ax.legend(frameon=True)
plt.tight_layout()
path1b = savefig("eda_log_popden_counts.png")
plt.show()
print("Saved:", path1b)

# 2) Original-scale popden ECDF on log-x (uses exp transform; robust to whether you used ln(x) vs ln(x+1))
fig, ax = plt.subplots(figsize=(7.2, 3.9))
orig = np.exp(vals_f.to_numpy())
orig = orig[np.isfinite(orig) & (orig > 0)]

# ECDF
x = np.sort(orig)
y = np.arange(1, len(x) + 1) / len(x)
ax.plot(x, y, color="#1f77b4", linewidth=2.2)
ax.set_xscale("log")
ax.set_title("Population density (original scale) — ECDF", fontweight="bold")
ax.set_xlabel("Population density (log scale)")
ax.set_ylabel("Cumulative share")
ax.grid(True, which="both", axis="x", alpha=0.25)
plt.tight_layout()
path2 = savefig("eda_popden_ecdf_logx.png")
plt.show()
print("Saved:", path2)

# 3) log(POBTOT) distribution (filtered)
fig, ax = plt.subplots(figsize=(7.2, 3.9))
vals_pop = df.loc[mask, "log_POBTOT"].replace([np.inf, -np.inf], np.nan).dropna()

sns.histplot(vals_pop, bins=80, stat="density", color="#2ca02c", alpha=0.45, ax=ax)
sns.kdeplot(vals_pop, color="#1b7f3a", linewidth=2.2, ax=ax)
ax.set_title("Population target distribution (ln-scale)", fontweight="bold")
ax.set_xlabel("ln(population)")
ax.set_ylabel("Density")
plt.tight_layout()
path3 = savefig("eda_log_population_distribution.png")
plt.show()
print("Saved:", path3)

# 4) Optional: group comparison if an urban/rural indicator exists
candidate_cols = [c for c in df.columns if any(k in c.lower() for k in ["urban", "rural", "ambito", "tipo", "clas", "ageb", "localidad", "loc"]) ]
print("Potential grouping columns (inspect if needed):", candidate_cols[:30])

group_col = None
for c in ["AMBITO", "ambito", "URBAN", "urban", "rural", "tipo", "TIPO", "tipo_loc", "TIPO_LOC"]:
    if c in df.columns:
        group_col = c
        break

if group_col is not None:
    g = df.loc[mask, [group_col, "log_popden"]].dropna()
    # Keep the most common categories if too many
    top = g[group_col].value_counts().head(6).index
    g = g[g[group_col].isin(top)]

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    sns.boxplot(data=g, x=group_col, y="log_popden", ax=ax, color="#8ecae6", fliersize=1)
    ax.set_title("ln(population density) by group", fontweight="bold")
    ax.set_xlabel(group_col)
    ax.set_ylabel("ln(population density)")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    path4 = savefig("eda_log_popden_by_group_boxplot.png")
    plt.show()
    print("Saved:", path4)
else:
    print("No obvious urban/rural grouping column found; skipping group plot.")


## 1.55 Original-scale population and density

Motivation for modeling on logs: heavy tails on linear axes vs $\log_{10}$ $x$-axis histograms (filtered sample).


In [ ]:
# Original-scale distributions (filtered sample only)

def _finite_positive(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    s = s.dropna()
    return s[s > 0]

if "pop_dens" in df.columns:
    popden_raw = _finite_positive(df.loc[mask, "pop_dens"])
    popden_source = "pop_dens (CSV)"
else:
    popden_raw = _finite_positive(np.exp(df.loc[mask, "log_popden"].astype(float)))
    popden_source = "exp(log_popden) (fallback)"

if "POBTOT" in df.columns:
    pob_raw = _finite_positive(df.loc[mask, "POBTOT"])
    pob_source = "POBTOT (CSV)"
else:
    pob_raw = _finite_positive(np.exp(df.loc[mask, "log_POBTOT"].astype(float)))
    pob_source = "exp(log_POBTOT) (fallback)"

print(f"Population density — source: {popden_source}")
print(popden_raw.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_string())
print()
print(f"Population — source: {pob_source}")
print(pob_raw.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_string())

# Linear panels: fixed bin edges up to a high quantile so the body of the distribution is visible
# (values above the cut are not placed in these bins; counts are noted on-panel).
q_lin = 0.995
xmax_pd = float(np.quantile(popden_raw, q_lin))
xmax_pob = float(np.quantile(pob_raw, q_lin))
n_over_pd = int((popden_raw > xmax_pd).sum())
n_over_pob = int((pob_raw > xmax_pob).sum())

fig, axes = plt.subplots(2, 2, figsize=(12.2, 6.6), sharex=False, sharey=False)

# --- population density: linear | log10 x ---
sns.histplot(
    popden_raw,
    bins=np.linspace(0, xmax_pd, 81),
    stat="count",
    color="#4e79a7",
    alpha=0.55,
    ax=axes[0, 0],
)
axes[0, 0].set_xlim(0, xmax_pd)
axes[0, 0].set_title(r"Population density — counts (linear $x$)", fontweight="bold")
axes[0, 0].set_xlabel(r"Population density (linear axis)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].text(
    0.02,
    0.98,
    f"{n_over_pd:,} obs. above {100 * q_lin:.1f}th percentile ({xmax_pd:.3g})",
    transform=axes[0, 0].transAxes,
    va="top",
    ha="left",
    fontsize=9,
)
axes[0, 0].grid(True, axis="y")
axes[0, 0].grid(False, axis="x")

sns.histplot(popden_raw, bins=80, stat="count", color="#4e79a7", alpha=0.55, ax=axes[0, 1])
axes[0, 1].set_xscale("log")
axes[0, 1].set_title(r"Population density — counts ($\log_{10}$ $x$-axis)", fontweight="bold")
axes[0, 1].set_xlabel(r"Population density ($\log_{10}$-scaled axis)")
axes[0, 1].set_ylabel("Count")
axes[0, 1].grid(True, axis="y")
axes[0, 1].grid(False, axis="x")

# --- population: linear | log10 x ---
sns.histplot(
    pob_raw,
    bins=np.linspace(0, xmax_pob, 81),
    stat="count",
    color="#59a14f",
    alpha=0.55,
    ax=axes[1, 0],
)
axes[1, 0].set_xlim(0, xmax_pob)
axes[1, 0].set_title(r"Total population — counts (linear $x$)", fontweight="bold")
axes[1, 0].set_xlabel(r"Population (linear axis)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].text(
    0.02,
    0.98,
    f"{n_over_pob:,} obs. above {100 * q_lin:.1f}th percentile ({xmax_pob:.3g})",
    transform=axes[1, 0].transAxes,
    va="top",
    ha="left",
    fontsize=9,
)
axes[1, 0].grid(True, axis="y")
axes[1, 0].grid(False, axis="x")

sns.histplot(pob_raw, bins=80, stat="count", color="#59a14f", alpha=0.55, ax=axes[1, 1])
axes[1, 1].set_xscale("log")
axes[1, 1].set_title(r"Total population — counts ($\log_{10}$ $x$-axis)", fontweight="bold")
axes[1, 1].set_xlabel(r"Population ($\log_{10}$-scaled axis)")
axes[1, 1].set_ylabel("Count")
axes[1, 1].grid(True, axis="y")
axes[1, 1].grid(False, axis="x")

plt.tight_layout()
path = savefig("eda_raw_popden_and_population_logx.png")
plt.show()
print("Saved:", path)


## 1.6 State-level breakdown

Counts per state and ln(popden) quantiles; figures saved under `report/figs/`.


In [ ]:
# State-level breakdown (targets)

if "CVE_ENT" not in df.columns:
    raise KeyError("Expected `CVE_ENT` (state code) column not found in df.")

# INEGI state code (01-32) to name
ent_to_state = {
    1: "Aguascalientes",
    2: "Baja California",
    3: "Baja California Sur",
    4: "Campeche",
    5: "Coahuila",
    6: "Colima",
    7: "Chiapas",
    8: "Chihuahua",
    9: "Ciudad de México",
    10: "Durango",
    11: "Guanajuato",
    12: "Guerrero",
    13: "Hidalgo",
    14: "Jalisco",
    15: "México",
    16: "Michoacán",
    17: "Morelos",
    18: "Nayarit",
    19: "Nuevo León",
    20: "Oaxaca",
    21: "Puebla",
    22: "Querétaro",
    23: "Quintana Roo",
    24: "San Luis Potosí",
    25: "Sinaloa",
    26: "Sonora",
    27: "Tabasco",
    28: "Tamaulipas",
    29: "Tlaxcala",
    30: "Veracruz",
    31: "Yucatán",
    32: "Zacatecas",
}

# Use the same filtered sample used for modeling
state_work = df.loc[mask, ["CVE_ENT", "log_popden", "log_POBTOT"]].copy()
state_work["log_popden"] = state_work["log_popden"].replace([np.inf, -np.inf], np.nan)
state_work["log_POBTOT"] = state_work["log_POBTOT"].replace([np.inf, -np.inf], np.nan)
state_work = state_work.dropna(subset=["log_popden", "log_POBTOT"])

# Normalize CVE_ENT to int where possible (handles strings like '09')
state_work["CVE_ENT_int"] = pd.to_numeric(state_work["CVE_ENT"], errors="coerce").astype("Int64")
state_work["State"] = state_work["CVE_ENT_int"].map(ent_to_state)
state_work["State"] = state_work["State"].fillna(state_work["CVE_ENT"].astype(str))

# Summary table
state_summary = (
    state_work.groupby("State")["log_popden"]
    .agg(
        n="count",
        mean="mean",
        p50="median",
        p90=lambda s: s.quantile(0.9),
        p99=lambda s: s.quantile(0.99),
    )
    .sort_values("n", ascending=False)
)
state_summary["share"] = state_summary["n"] / state_summary["n"].sum()

# Publication-friendly ordering: largest n first
order_states = state_summary.index.tolist()

display(state_summary)

# Plot 1: counts per state
fig, ax = plt.subplots(figsize=(10.0, 4.2))
ax.bar(range(len(order_states)), state_summary.loc[order_states, "n"], color="#4e79a7", alpha=0.92)
ax.set_title("Filtered sample size by state", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Number of polygons")
ax.set_xticks(range(len(order_states)))
ax.set_xticklabels(order_states, rotation=60, ha="right")
ax.grid(True, axis="y")
ax.grid(False, axis="x")
plt.tight_layout()
path = savefig("eda_state_counts_filtered.png")
plt.show()
print("Saved:", path)

# Plot 2: median + tails by state (dot + whiskers)
fig, ax = plt.subplots(figsize=(10.0, 4.8))
xs = np.arange(len(order_states))
p50 = state_summary.loc[order_states, "p50"].to_numpy()
p90 = state_summary.loc[order_states, "p90"].to_numpy()
p99 = state_summary.loc[order_states, "p99"].to_numpy()

ax.vlines(xs, p50, p90, color="#9aa0a6", linewidth=2.0, alpha=0.9)
ax.vlines(xs, p90, p99, color="#c0c0c0", linewidth=2.0, alpha=0.8)
ax.scatter(xs, p50, color="#1f77b4", s=28, zorder=3, label="Median")
ax.scatter(xs, p90, color="#ff7f0e", s=20, zorder=3, label="90th pct.")
ax.scatter(xs, p99, color="#d62728", s=18, zorder=3, label="99th pct.")

ax.set_title("ln(population density) by state (filtered)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("ln(population density)")
ax.set_xticks(xs)
ax.set_xticklabels(order_states, rotation=60, ha="right")
ax.grid(True, axis="y")
ax.grid(False, axis="x")
ax.legend(frameon=True, ncol=3, loc="upper right")
plt.tight_layout()
path = savefig("eda_state_log_popden_quantiles.png")
plt.show()
print("Saved:", path)


## 1.7 Polygon subtype (AGEB vs localidad proxy)

Heuristic labels from geographic keys and population; boxplot and summary table.


In [ ]:
# Derive an approximate unit/type indicator from available keys
# (Useful for EDA only; the official INEGI definitions are more nuanced.)

key_cols = [c for c in ["CVE_ENT", "CVE_MUN", "CVE_LOC", "CVE_AGEB", "CVEGEO"] if c in df.columns]
print("Key columns present:", key_cols)

work = df.loc[mask, ["log_popden", "log_POBTOT"] + key_cols].copy()
work["pop_est"] = np.exp(work["log_POBTOT"].astype(float))

unit_type = pd.Series("Unknown", index=work.index)

if "CVE_AGEB" in work.columns:
    unit_type = np.where(work["CVE_AGEB"].notna() & (work["CVE_AGEB"].astype(str).str.len() > 0), "AGEB", "Localidad")
else:
    # Fallback: treat rows without a locality code as AGEB-like
    if "CVE_LOC" in work.columns:
        unit_type = np.where(work["CVE_LOC"].notna() & (work["CVE_LOC"].astype(str).str.len() > 0), "Localidad", "AGEB")

work["unit_type"] = unit_type

# Approx urban/rural split
# - AGEB: if CVE_LOC exists, treat AGEBs with locality code as Urban AGEB, otherwise Rural AGEB.
# - Localidad: use 2,500 population threshold on back-transformed population as a proxy.

subtype = pd.Series("Unknown", index=work.index)
if "CVE_LOC" in work.columns:
    is_ageb = work["unit_type"].eq("AGEB")
    has_loc = work["CVE_LOC"].notna() & (work["CVE_LOC"].astype(str).str.len() > 0)
    subtype.loc[is_ageb & has_loc] = "Urban AGEB"
    subtype.loc[is_ageb & ~has_loc] = "Rural AGEB"

is_loc = work["unit_type"].eq("Localidad")
subtype.loc[is_loc & (work["pop_est"] >= 2500)] = "Urban Localidad (proxy)"
subtype.loc[is_loc & (work["pop_est"] < 2500)] = "Rural Localidad (proxy)"
work["subtype"] = subtype

print(work["subtype"].value_counts(dropna=False).head(10))

# Publication-ready comparison plot
plot_df = work.loc[work["subtype"].ne("Unknown"), ["subtype", "log_popden"]].dropna()
order = [
    "Urban AGEB",
    "Rural AGEB",
    "Urban Localidad (proxy)",
    "Rural Localidad (proxy)",
]
order = [o for o in order if o in set(plot_df["subtype"]) ]

fig, ax = plt.subplots(figsize=(7.4, 4.4))
sns.boxplot(
    data=plot_df,
    x="subtype",
    y="log_popden",
    order=order,
    palette=["#1f77b4", "#ff7f0e", "#2ca02c", "#7f7f7f"][: len(order)],
    fliersize=1,
    ax=ax,
)
ax.set_title("ln(population density) by polygon subtype", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("ln(population density)")
ax.tick_params(axis="x", rotation=18)
plt.tight_layout()
path = savefig("eda_log_popden_by_subtype_boxplot.png")
plt.show()
print("Saved:", path)

# Simple summary table (publication-friendly)
summary_tbl = (
    plot_df.groupby("subtype")["log_popden"]
    .agg(n="count", mean="mean", p50="median", p90=lambda s: s.quantile(0.9), p99=lambda s: s.quantile(0.99))
    .reindex(order)
)
display(summary_tbl)
